# L5 demo: a 4-stage SECOM pipeline, three ways

This notebook builds the same four-stage batch pipeline (**ingest → clean → impute →
aggregate**) on the UCI SECOM dataset three times: once in pandas, once in Polars
lazy, and once in Dask across partitions. Each version is a single reproducible
function so it can be timed and rerun. The pandas and Polars runs should agree on
every number; the Dask run exists to make the out-of-core execution model concrete,
not because this dataset needs it.

SECOM is 1,567 semiconductor manufacturing runs with 590 sensor/process
measurements each, heavy missingness, several constant columns, and a pass/fail
label. It is small enough to fit in memory on a laptop, which is itself part of
the lesson: watch for the point where Dask's overhead outweighs its benefit.

## Fetch and cache the data

The raw files are cached under `.cache/` on first run, exactly as in the L1 and L3
demos, so re-running this notebook does not re-download anything.

In [ ]:
import io
import urllib.request
import warnings
import zipfile
from pathlib import Path

import pandas as pd
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
# A 590-wide dataframe built by concatenation trips pandas' block-fragmentation
# heuristic on every later column write. Harmless here; silenced so the signal
# is not lost in the noise for the operations that actually matter.

CACHE = Path('.cache')
CACHE.mkdir(exist_ok=True)

DATA_FILE = CACHE / 'secom.data'
LABELS_FILE = CACHE / 'secom_labels.data'
URL = 'https://archive.ics.uci.edu/static/public/179/secom.zip'

if not DATA_FILE.exists():
    print('downloading', URL)
    with urllib.request.urlopen(URL) as r:
        payload = r.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as z:
        DATA_FILE.write_bytes(z.read('secom.data'))
        LABELS_FILE.write_bytes(z.read('secom_labels.data'))


## Stage 1: ingest

`secom.data` is 590 whitespace-separated columns with no header and `NaN` for
missing values. `secom_labels.data` carries the pass/fail label and a
collection timestamp per run, in the same row order. We tag each run with a
`run_id` so every later stage can be traced back to a specific wafer run.

In [ ]:
import numpy as np
import pandas as pd

def ingest_pandas(data_file=DATA_FILE, labels_file=LABELS_FILE):
    X = pd.read_csv(data_file, sep=r'\s+', header=None, na_values='NaN')
    X.columns = [f'sensor_{i}' for i in range(X.shape[1])]

    lab = pd.read_csv(labels_file, sep=r'\s+', header=None,
                       names=['label', 'date', 'time'])
    lab['ts'] = pd.to_datetime(lab['date'] + ' ' + lab['time'],
                                format='%d/%m/%Y %H:%M:%S')

    raw = pd.concat([lab[['ts', 'label']], X], axis=1)
    raw = pd.concat([pd.Series(np.arange(len(raw)), name='run_id'), raw], axis=1)
    return raw.copy()  # one contiguous block; avoids fragmentation on later column writes

raw = ingest_pandas()
sensor_cols = [c for c in raw.columns if c.startswith('sensor_')]
print(raw.shape, 'runs x columns;', len(sensor_cols), 'sensors')
raw.head(3)


## Stage 2: clean

Drop columns that are constant (zero information) or missing beyond a
threshold, computed on the **training split only** -- fitting this on the full
dataset, test rows included, is exactly the leakage bug A3 asks you to avoid.

In [ ]:
def clean_pandas(df, sensor_cols, missing_frac=0.4):
    """Return the trimmed frame and the columns it dropped."""
    nunique = df[sensor_cols].nunique(dropna=True)
    constant = nunique[nunique <= 1].index
    miss = df[sensor_cols].isna().mean()
    high_missing = miss[miss > missing_frac].index
    drop_cols = sorted(set(constant) | set(high_missing))
    return df.drop(columns=drop_cols), drop_cols

n_train = int(len(raw) * 0.8)
train, test = raw.iloc[:n_train].copy(), raw.iloc[n_train:].copy()

train_clean, dropped = clean_pandas(train, sensor_cols)
test_clean = test.drop(columns=dropped)
keep_cols = [c for c in train_clean.columns if c.startswith('sensor_')]
print(f'{len(dropped)} columns dropped (constant or >40% missing), {len(keep_cols)} remain')


## Stage 3: impute

Fill missing sensor readings with the **training mean**, then apply that same,
frozen statistic to the test split. Computing the mean on the test rows too, or
on the whole dataset before splitting, is the single most common way this kind
of pipeline leaks the future into the past.

In [ ]:
def impute_pandas(reference, target, cols):
    means = reference[cols].mean()
    out = target.copy()
    out[cols] = out[cols].fillna(means)
    return out

train_imputed = impute_pandas(train_clean, train_clean, keep_cols)
test_imputed = impute_pandas(train_clean, test_clean, keep_cols)
assert train_imputed[keep_cols].isna().sum().sum() == 0
assert test_imputed[keep_cols].isna().sum().sum() == 0


## Stage 4: aggregate and persist

A daily mean per sensor, cached to Parquet. Writing the same input twice
produces the same file, which is the whole point of keeping every stage a
pure function of its input: the pipeline is idempotent by construction, not
by careful bookkeeping.

In [ ]:
def aggregate_pandas(df, cols):
    return df.assign(day=df['ts'].dt.date).groupby('day')[cols].mean()

daily = aggregate_pandas(train_imputed, keep_cols)

out_path = CACHE / 'train_imputed.parquet'
train_imputed.to_parquet(out_path, index=False)
print('wrote', out_path, train_imputed.shape)
daily.head()


## The same four stages in Polars, lazy

One lazy query, described end to end and only executed at `.collect()`. Polars
computes the per-column statistics clean and impute both need -- distinct
count, missing count, and mean -- in a **single pass** over the training
partition, instead of the three separate passes the pandas version above
makes.

In [ ]:
import polars as pl

def secom_polars_pipeline(data_file=DATA_FILE, labels_file=LABELS_FILE,
                           missing_frac=0.4, train_frac=0.8):
    X = pl.read_csv(data_file, separator=' ', has_header=False, null_values=['NaN'])
    X.columns = [f'sensor_{i}' for i in range(X.width)]

    lab = pl.read_csv(labels_file, separator=' ', has_header=False,
                       new_columns=['label', 'date', 'time'])
    lab = lab.with_columns(
        (pl.col('date') + ' ' + pl.col('time'))
        .str.strptime(pl.Datetime, '%d/%m/%Y %H:%M:%S').alias('ts')
    )

    lf = X.with_row_index('run_id').with_columns([lab['ts'], lab['label']]).lazy()
    sensor_cols = [c for c in lf.collect_schema().names() if c.startswith('sensor_')]

    n_total = lf.select(pl.len()).collect().item()
    n_train = int(n_total * train_frac)
    train_lf = lf.slice(0, n_train)
    test_lf = lf.slice(n_train, n_total - n_train)

    # clean() and impute() share this one pass over the training partition.
    stats = train_lf.select(
        [pl.col(c).drop_nulls().n_unique().alias(f'{c}__nunique') for c in sensor_cols]
        + [pl.col(c).null_count().alias(f'{c}__nmiss') for c in sensor_cols]
        + [pl.col(c).mean().alias(f'{c}__mean') for c in sensor_cols]
        + [pl.len().alias('__n')]
    ).collect()
    n_train_rows = stats['__n'][0]

    keep = [
        c for c in sensor_cols
        if stats[f'{c}__nunique'][0] > 1
        and stats[f'{c}__nmiss'][0] / n_train_rows <= missing_frac
    ]
    means = {c: stats[f'{c}__mean'][0] for c in keep}

    def clean_and_impute(lfr):
        return (
            lfr.select(['run_id', 'ts', 'label'] + keep)
            .with_columns([pl.col(c).fill_null(means[c]) for c in keep])
        )

    train_out = clean_and_impute(train_lf)
    test_out = clean_and_impute(test_lf)
    daily_out = (
        train_out.with_columns(pl.col('ts').dt.date().alias('day'))
        .group_by('day').agg([pl.col(c).mean() for c in keep]).sort('day')
    )
    return train_out.collect(), test_out.collect(), daily_out.collect(), keep

train_pl, test_pl, daily_pl, keep_pl = secom_polars_pipeline()
print(f'{len(keep_pl)} sensors kept, {590 - len(keep_pl)} dropped -- matches the pandas run')
assert set(keep_pl) == set(keep_cols)


:::{admonition} Common pitfall
:class: warning

`clean_pandas` above uses `nunique(dropna=True)` to find constant columns, and
that `dropna` is load-bearing. Polars' `Series.n_unique()` counts a null as a
distinct value by default, so a column that is one constant value plus a
scatter of missing readings reports **two** unique values, not one, and
survives a naive port of the pandas rule. The fix is `pl.col(c).drop_nulls().n_unique()`,
used above. Porting a pandas idiom to Polars column by column, rather than
rereading what each call actually does, is exactly how a "clean" step quietly
stops cleaning.
:::

## Benchmark: pandas vs. Polars, the whole pipeline

Both functions read the raw files and run all four stages, so this times the
thing you actually care about end to end, not one operation in isolation.
Run it yourself; the ratio you get depends on your machine, but the direction
should not surprise you now that both pipelines are one pass over the data
rather than pandas' several.

In [ ]:
import time

def secom_pandas_pipeline(data_file=DATA_FILE, labels_file=LABELS_FILE,
                           missing_frac=0.4, train_frac=0.8):
    raw = ingest_pandas(data_file, labels_file)
    cols = [c for c in raw.columns if c.startswith('sensor_')]
    n_train = int(len(raw) * train_frac)
    train, test = raw.iloc[:n_train].copy(), raw.iloc[n_train:].copy()
    train_c, dropped = clean_pandas(train, cols, missing_frac)
    test_c = test.drop(columns=dropped)
    keep = [c for c in train_c.columns if c.startswith('sensor_')]
    train_i = impute_pandas(train_c, train_c, keep)
    test_i = impute_pandas(train_c, test_c, keep)
    daily_i = aggregate_pandas(train_i, keep)
    return train_i, test_i, daily_i, keep

def timed(fn, repeats=3):
    return min(_time_once(fn) for _ in range(repeats))

def _time_once(fn):
    t0 = time.perf_counter()
    fn()
    return time.perf_counter() - t0

t_pandas = timed(secom_pandas_pipeline)
t_polars = timed(secom_polars_pipeline)
print(f'pandas: {t_pandas * 1000:7.1f} ms')
print(f'polars: {t_polars * 1000:7.1f} ms   ({t_pandas / t_polars:.1f}x)')


## Dask: the same pipeline, spread across partitions

This is a concepts demo, not a recommendation: 1,567 rows fit in RAM many
times over, and a real Dask deployment earns its keep on data that does not.
We split the training rows into partitions and run the pipeline as Dask
dataframe operations, so the mechanism -- a task graph over chunks, only
executed at `.compute()` -- is visible on a single laptop.

The first version below is the direct, naive port: reuse the pandas
`nunique(dropna=True)` idiom to find constant columns. Time it once before
reading on, because the result is the point.

In [ ]:
import dask.dataframe as dd

def secom_dask_pipeline_naive(raw, sensor_cols, npartitions=8,
                               missing_frac=0.4, train_frac=0.8):
    n_train = int(len(raw) * train_frac)
    train = raw.iloc[:n_train]
    ddf = dd.from_pandas(train, npartitions=npartitions)

    nunique = ddf[sensor_cols].nunique(dropna=True).compute()   # <- watch this one
    miss = ddf[sensor_cols].isna().mean().compute()
    constant = nunique[nunique <= 1].index
    high_missing = miss[miss > missing_frac].index
    keep = [c for c in sensor_cols if c not in set(constant) | set(high_missing)]
    return keep

t0 = time.perf_counter()
keep_naive = secom_dask_pipeline_naive(raw, sensor_cols, npartitions=8)
print(f'{time.perf_counter() - t0:.1f} s for the nunique-based version -- on 1,567 rows.')


That is not a typo, and it is not this dataset being unusually hard. Dask's
per-column `nunique` builds a distinct-value computation for every one of 590
columns as a separate piece of the task graph, and the bookkeeping overhead
of that graph, not the actual arithmetic, is what you are waiting on. It is a
real, documented sharp edge, and the fix is to stop asking Dask to count
distinct values at all: a numeric column is constant exactly when its
standard deviation is zero, and `std` is a single, cheap, already-parallel
reduction.

In [ ]:
def secom_dask_pipeline(raw, sensor_cols, npartitions=8,
                         missing_frac=0.4, train_frac=0.8):
    n_train = int(len(raw) * train_frac)
    train = raw.iloc[:n_train]
    ddf = dd.from_pandas(train, npartitions=npartitions)

    std = ddf[sensor_cols].std().compute()
    miss = ddf[sensor_cols].isna().mean().compute()
    constant = std[(std == 0) | std.isna()].index
    high_missing = miss[miss > missing_frac].index
    keep = [c for c in sensor_cols if c not in set(constant) | set(high_missing)]
    means = ddf[keep].mean().compute()

    def process_partition(part):
        out = part[['run_id', 'ts'] + keep].fillna(means)
        return out.assign(day=out['ts'].dt.date).copy()

    meta = process_partition(train.iloc[:2])
    processed = ddf.map_partitions(process_partition, meta=meta)
    daily = processed.groupby('day')[keep].mean().compute()
    return daily, keep

t0 = time.perf_counter()
daily_dask, keep_dask = secom_dask_pipeline(raw, sensor_cols, npartitions=8)
t_dask = time.perf_counter() - t0
print(f'dask (std-based, {8} partitions): {t_dask * 1000:7.1f} ms')
print(f'pandas, for comparison:           {t_pandas * 1000:7.1f} ms')
assert set(keep_dask) == set(keep_cols)


## What the timings are telling you

The corrected Dask run is fast enough to sit through, but it is still slower
than plain pandas on this data, and that gap is the lesson, not a bug to
chase. Every Dask operation pays for building and scheduling a task graph
across partitions before a single number is computed. On a table that fits in
memory, that overhead is pure cost, because there was never anything to
parallelize across machines in the first place. The break-even point is data
that does not fit in your laptop's RAM, or a computation heavy enough that
spreading it across cores or machines pays for the scheduling. SECOM at 1,567
rows is nowhere near that point; it is here so the mechanism -- partitions, a
lazy task graph, `.compute()` -- is something you have seen work, not just
read about, before you meet it at a scale where it matters.

Full notes, with the pandas/Polars/Dask trade-offs and the rest of the
argument: [`../notes.md`](notes.md).